# Z1 · M4: przeszukiwanie przestrzeni stanów

Druga część zadania Z1, razem z notebookiem M3 (`lab_wnioskowanie_w_przod.ipynb`) o systemie regułowym do wykrywania oszustw kartowych. Tu: BFS, DFS i A* w labiryncie, wnioskowanie wstecz z M3 zapisane jako DFS oraz miniatura Tree of Thoughts na regułach z projektu.

Schemat ogólny: frontier z węzłem startowym, wybór węzła (FIFO dla BFS, LIFO dla DFS, najmniejsze f dla A*), test celu, rozwinięcie następników, dodanie nieodwiedzonych do frontier. Algorytmy różnią się tylko sposobem wyboru węzła.

## G1. Przestrzeń stanów labiryntu

Stan to para (wiersz, kolumna), akcje to ruchy góra, dół, lewo, prawo. Liczę stany i średni współczynnik rozgałęzienia.

In [ ]:
import matplotlib.pyplot as plt

# S - start, G - cel, # - ściana
LABIRYNT = [
    'S..#......',
    '.#.#.####.',
    '.#...#....',
    '.####.#.#.',
    '......#.#G',
]

def znajdz(lab, znak):
    for r, wiersz in enumerate(lab):
        if znak in wiersz:
            return (r, wiersz.index(znak))

def sasiedzi(lab, stan):
    r, c = stan
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:   # góra, dół, lewo, prawo
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(lab) and 0 <= nc < len(lab[0]) and lab[nr][nc] != '#':
            yield (nr, nc)

START, CEL = znajdz(LABIRYNT, 'S'), znajdz(LABIRYNT, 'G')
print('start:', START, 'cel:', CEL, 'sąsiedzi startu:', list(sasiedzi(LABIRYNT, START)))

stany = [(r, c) for r in range(len(LABIRYNT)) for c in range(len(LABIRYNT[0])) if LABIRYNT[r][c] != '#']
b = sum(len(list(sasiedzi(LABIRYNT, s))) for s in stany) / len(stany)
print('liczba stanów:', len(stany), '| średni współczynnik rozgałęzienia:', round(b, 2))

def rysuj(lab, sciezka=None, tytul=''):
    plt.figure(figsize=(5, 2.5))
    plt.imshow([[c == '#' for c in w] for w in lab], cmap='Greys')
    if sciezka:
        plt.plot([c for r, c in sciezka], [r for r, c in sciezka], 'r-o', markersize=3)
    s, g = znajdz(lab, 'S'), znajdz(lab, 'G')
    plt.text(s[1], s[0], 'S', ha='center', va='center', color='green', fontweight='bold')
    plt.text(g[1], g[0], 'G', ha='center', va='center', color='blue', fontweight='bold')
    plt.title(tytul)
    plt.xticks([])
    plt.yticks([])
    plt.show()

rysuj(LABIRYNT, tytul='Labirynt przykładowy')


## G2. BFS i DFS

Jeden kod dla obu algorytmów. BFS bierze węzeł z początku frontier (`popleft`), DFS z końca (`pop`). Liczniki: węzły rozwinięte i maksymalny rozmiar frontier (miara pamięci).

In [ ]:
from collections import deque

def szukaj(lab, strategia='bfs'):
    start, cel = znajdz(lab, 'S'), znajdz(lab, 'G')
    frontier = deque([start])
    rodzic = {start: None}          # to jest też zbiór odwiedzonych
    rozwiniete = 0
    max_frontier = 1
    while frontier:
        stan = frontier.popleft() if strategia == 'bfs' else frontier.pop()
        rozwiniete += 1
        if stan == cel:
            sciezka = []
            while stan is not None:
                sciezka.append(stan)
                stan = rodzic[stan]
            return {'sciezka': sciezka[::-1], 'rozwiniete': rozwiniete, 'max_frontier': max_frontier}
        for n in sasiedzi(lab, stan):
            if n not in rodzic:
                rodzic[n] = stan
                frontier.append(n)
        max_frontier = max(max_frontier, len(frontier))
    return {'sciezka': None, 'rozwiniete': rozwiniete, 'max_frontier': max_frontier}

for strat in ['bfs', 'dfs']:
    w = szukaj(LABIRYNT, strat)
    print(strat.upper(), '| długość ścieżki:', len(w['sciezka']) - 1,
          '| rozwinięte:', w['rozwiniete'], '| max frontier:', w['max_frontier'])
    rysuj(LABIRYNT, w['sciezka'], strat.upper())


## G3. Porównanie algorytmów

Cztery labirynty: przykładowy, otwarty (bez ścian), pułapka dla DFS (duża otwarta sala obok krótkiej drogi do celu) oraz wąski korytarz.

In [ ]:
LABIRYNTY = {
    'przykladowy': LABIRYNT,
    'otwarty': [
        'S.......',
        '........',
        '........',
        '........',
        '........',
        '........',
        '........',
        '.......G',
    ],
    'pulapka': [
        '##########',
        '#G.S.....#',
        '#.#......#',
        '#.#......#',
        '#.#......#',
        '#.#......#',
        '#........#',
        '##########',
    ],
    'korytarz': [
        'S.#.......',
        '#.#.#####.',
        '#.#.#...#.',
        '#.#.#.#.#.',
        '#...#.#...',
        '#####.###G',
    ],
}

# sprawdzenie, że wszystkie labirynty mają rozwiązanie
for nazwa, lab in LABIRYNTY.items():
    assert szukaj(lab, 'bfs')['sciezka'] is not None, nazwa

import pandas as pd

def porownaj(labirynty, algorytmy):
    wiersze = []
    for nazwa_lab, lab in labirynty.items():
        for nazwa_alg, alg in algorytmy.items():
            w = alg(lab)
            wiersze.append({'labirynt': nazwa_lab, 'algorytm': nazwa_alg,
                            'dlugosc_sciezki': len(w['sciezka']) - 1,
                            'rozwiniete': w['rozwiniete'], 'max_frontier': w['max_frontier']})
    return pd.DataFrame(wiersze)

def wykres(tabela):
    fig, osie = plt.subplots(1, 3, figsize=(13, 3.5))
    nazwy = ['Długość ścieżki', 'Rozwinięte węzły', 'Maks. rozmiar frontier']
    for os_, kol, tytul in zip(osie, ['dlugosc_sciezki', 'rozwiniete', 'max_frontier'], nazwy):
        tabela.pivot(index='labirynt', columns='algorytm', values=kol).plot.bar(ax=os_, rot=0)
        os_.set_title(tytul)
        os_.set_xlabel('')
    plt.tight_layout()
    plt.show()

ALGORYTMY = {'BFS': lambda lab: szukaj(lab, 'bfs'), 'DFS': lambda lab: szukaj(lab, 'dfs')}
tabela = porownaj(LABIRYNTY, ALGORYTMY)
print(tabela.to_string(index=False))
wykres(tabela)


## B1. A* z heurystyką Manhattan

Węzły ze stałym kosztem kroku 1. Priorytet to f = g + waga * h, gdzie h to odległość Manhattan do celu. Dla waga = 1 heurystyka jest dopuszczalna. Większa waga przeszacowuje odległość, co sprawdzam niżej.

In [ ]:
import heapq

def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def astar(lab, waga=1):
    start, cel = znajdz(lab, 'S'), znajdz(lab, 'G')
    licznik = 0                       # rozstrzyga remisy w kolejce
    frontier = [(waga * manhattan(start, cel), 0, licznik, start)]
    rodzic = {start: None}
    koszt = {start: 0}
    zamkniete = set()
    rozwiniete = 0
    max_frontier = 1
    while frontier:
        f, g, _, stan = heapq.heappop(frontier)
        if stan in zamkniete:
            continue
        zamkniete.add(stan)
        rozwiniete += 1
        if stan == cel:
            sciezka = []
            while stan is not None:
                sciezka.append(stan)
                stan = rodzic[stan]
            return {'sciezka': sciezka[::-1], 'rozwiniete': rozwiniete, 'max_frontier': max_frontier}
        for n in sasiedzi(lab, stan):
            ng = g + 1
            if n not in koszt or ng < koszt[n]:
                koszt[n] = ng
                rodzic[n] = stan
                licznik += 1
                heapq.heappush(frontier, (ng + waga * manhattan(n, cel), ng, licznik, n))
        max_frontier = max(max_frontier, len(frontier))
    return {'sciezka': None, 'rozwiniete': rozwiniete, 'max_frontier': max_frontier}

ALGORYTMY['A*'] = astar
tabela = porownaj(LABIRYNTY, ALGORYTMY)
print(tabela.to_string(index=False))
wykres(tabela)

# A* z dopuszczalną heurystyką musi dać taką samą długość jak BFS
for nazwa, lab in LABIRYNTY.items():
    assert len(astar(lab)['sciezka']) == len(szukaj(lab, 'bfs')['sciezka']), nazwa
print('A* (waga=1) ma tę samą długość ścieżki co BFS na wszystkich labiryntach')

# przeszacowana heurystyka: na labiryntach z G3 nic się nie zmienia, więc osobny labirynt
for nazwa, lab in LABIRYNTY.items():
    dl = [len(astar(lab, w)['sciezka']) - 1 for w in (1, 3, 10)]
    print(nazwa, '| długość ścieżki dla waga 1, 3, 10:', dl)

LABIRYNT_H = [
    'S.......##',
    '.....#....',
    '.#....#.##',
    '.#.##.#...',
    '.......##.',
    '.....###..',
    '.....#.##G',
]
dl = [len(astar(LABIRYNT_H, w)['sciezka']) - 1 for w in (1, 3, 10)]
print('labirynt_h | długość ścieżki dla waga 1, 3, 10:', dl, '| BFS:', len(szukaj(LABIRYNT_H, 'bfs')['sciezka']) - 1)


## Reguły z części M3

B2 i P1 korzystają z bazy reguł o ryzyku transakcji kartowych i z wnioskowania wstecz z notebooka M3. Poniżej kopia potrzebnych fragmentów, żeby ten notebook działał samodzielnie.

In [ ]:
REGULY = [
    {'id': 'R1', 'if': {'wysoka_kwota', 'kwota_nietypowa_dla_karty'}, 'then': 'anomalia_kwoty', 'prio': 1},
    {'id': 'R2', 'if': {'duza_odleglosc_adresow', 'inny_kraj'}, 'then': 'anomalia_lokalizacji', 'prio': 1},
    {'id': 'R3', 'if': {'nowe_urzadzenie', 'rozne_domeny_email'}, 'then': 'anomalia_tozsamosci', 'prio': 1},
    {'id': 'R4', 'if': {'wysokie_liczniki_c', 'krotki_odstep_od_poprzedniej'}, 'then': 'anomalia_aktywnosci', 'prio': 1},
    {'id': 'R5', 'if': {'anomalia_kwoty', 'anomalia_lokalizacji'}, 'then': 'wysokie_ryzyko', 'prio': 3},
    {'id': 'R6', 'if': {'anomalia_aktywnosci', 'anomalia_tozsamosci'}, 'then': 'wysokie_ryzyko', 'prio': 3},
    {'id': 'R8', 'if': {'flagi_m_niezgodne', 'pierwsza_transakcja_karty'}, 'then': 'srednie_ryzyko', 'prio': 2},
    {'id': 'R10', 'if': {'anomalia_lokalizacji', 'transakcja_nocna'}, 'then': 'srednie_ryzyko', 'prio': 2},
    {'id': 'R11', 'if': {'stala_historia_karty', 'tozsamosc_zweryfikowana'}, 'then': 'niskie_ryzyko', 'prio': 1},
    {'id': 'R13', 'if': {'wiele_malych_transakcji', 'pierwsza_transakcja_karty'}, 'then': 'podejrzenie_testowania_karty', 'prio': 2},
    {'id': 'R7', 'if': {'wysokie_ryzyko'}, 'then': 'zalecenie_zablokuj_do_weryfikacji', 'prio': 1},
    {'id': 'R9', 'if': {'srednie_ryzyko'}, 'then': 'zalecenie_eskaluj_kontakt_z_klientem', 'prio': 1},
    {'id': 'R12', 'if': {'niskie_ryzyko'}, 'then': 'zalecenie_zatwierdz', 'prio': 1},
    {'id': 'R14', 'if': {'podejrzenie_testowania_karty'}, 'then': 'zalecenie_zablokuj_do_weryfikacji', 'prio': 1},
    {'id': 'R15', 'if': {'wysokie_ryzyko', 'tozsamosc_zweryfikowana'}, 'then': 'zalecenie_eskaluj_kontakt_z_klientem', 'prio': 2},
]

WEJSCIA = {'wysoka_kwota', 'kwota_nietypowa_dla_karty', 'duza_odleglosc_adresow', 'inny_kraj',
           'nowe_urzadzenie', 'rozne_domeny_email', 'wysokie_liczniki_c', 'krotki_odstep_od_poprzedniej',
           'flagi_m_niezgodne', 'pierwsza_transakcja_karty', 'transakcja_nocna', 'stala_historia_karty',
           'tozsamosc_zweryfikowana', 'wiele_malych_transakcji'}

TRANSAKCJA = {'wysoka_kwota', 'kwota_nietypowa_dla_karty', 'duza_odleglosc_adresow', 'inny_kraj',
              'tozsamosc_zweryfikowana'}


def backward_chain(cel, reguly, fakty, odwiedzone=frozenset()):
    # wnioskowanie wstecz z M3 (wersja rekurencyjna, bez śladu); cykle są odrzucane
    if cel in fakty:
        return True
    if cel in odwiedzone:
        return False
    for r in reguly:
        if r['then'] == cel and all(backward_chain(p, reguly, fakty, odwiedzone | {cel}) for p in sorted(r['if'])):
            return True
    return False


def brakujace_fakty(cel, reguly, fakty, odwiedzone=frozenset()):
    # najmniejszy zbiór brakujących faktów wejściowych potrzebny do celu
    if cel in fakty or cel in odwiedzone:
        return set()
    regs = [r for r in reguly if r['then'] == cel]
    if not regs:
        return {cel}
    kandydaci = []
    for r in regs:
        braki = set()
        for p in r['if']:
            braki |= brakujace_fakty(p, reguly, fakty, odwiedzone | {cel})
        kandydaci.append(braki)
    return min(kandydaci, key=len)


## B2. Backward chaining jako DFS

Wnioskowanie wstecz z M3 to DFS. Stan to lista celów do udowodnienia. Następniki powstają przez zastąpienie pierwszego celu przesłankami reguły, która go wyprowadza (albo przez jego usunięcie, gdy jest faktem). Stan pusty to stan docelowy. Frontier to stos, jak w DFS z G2.

In [ ]:
from itertools import combinations

def backward_jako_dfs(cel, reguly, fakty):
    fakty = set(fakty)
    start = (cel,)
    stos = [start]
    odwiedzone = {start}
    rozwiniete = 0
    while stos:
        stan = stos.pop()
        rozwiniete += 1
        if not stan:
            return True, rozwiniete
        pierwszy, reszta = stan[0], stan[1:]
        if pierwszy in fakty:
            nastepniki = [reszta]
        else:
            nastepniki = [tuple(sorted(r['if'])) + reszta for r in reguly if r['then'] == pierwszy]
        for n in reversed(nastepniki):     # pierwsza reguła rozwijana jako pierwsza
            if n not in odwiedzone:
                odwiedzone.add(n)
                stos.append(n)
    return False, rozwiniete

# ten sam wynik co wersja rekurencyjna dla każdego celu i każdego zbioru do 3 faktów wejściowych
cele = sorted({r['then'] for r in REGULY} | WEJSCIA)
liczba = 0
for k in range(4):
    for fakty in combinations(sorted(WEJSCIA), k):
        for cel in cele:
            assert backward_jako_dfs(cel, REGULY, fakty)[0] == backward_chain(cel, REGULY, set(fakty)), (fakty, cel)
        liczba += 1
print('backward_jako_dfs zgadza się z backward_chain dla', liczba, 'zbiorów faktów i', len(cele), 'celów')

for nazwa, fakty in [('transakcja z nietypową kwotą i lokalizacją', TRANSAKCJA), ('tylko wysoka kwota', {'wysoka_kwota'})]:
    ok, n = backward_jako_dfs('zalecenie_zablokuj_do_weryfikacji', REGULY, fakty)
    print(nazwa, '| cel udowodniony:', ok, '| rozwinięte stany:', n)


## P1. Tree of Thoughts w miniaturze

Węzeł to zbiór cech transakcji potwierdzonych do tej pory. Następnik powstaje przez sprawdzenie kolejnej cechy (jedna myśl). Funkcja oceny to liczba faktów, których jeszcze brakuje do udowodnienia hipotezy `wysokie_ryzyko` (z `brakujace_fakty`). Przeszukiwanie idzie poziomami (BFS), a z ograniczoną szerokością zostają tylko najlepsze stany.

W prawdziwym Tree of Thoughts generowanie myśli i ocenę robi LLM. Tu to zwykłe funkcje, a wynik sprawdzenia cechy bierzemy z założonego stanu faktycznego (`PRAWDA`).

In [ ]:
CEL_TOT = 'wysokie_ryzyko'
PRAWDA = {'wysoka_kwota', 'kwota_nietypowa_dla_karty', 'duza_odleglosc_adresow', 'inny_kraj',
          'transakcja_nocna', 'nowe_urzadzenie', 'rozne_domeny_email', 'wysokie_liczniki_c', 'flagi_m_niezgodne'}

def generuj(stan):
    # myśl: sprawdzenie jednej z jeszcze nieznanych cech, tu zawsze potwierdzonej
    return [stan | {f} for f in sorted(PRAWDA - stan)]

def ocen(stan):
    return -len(brakujace_fakty(CEL_TOT, REGULY, stan))

def cel_osiagniety(stan):
    return backward_chain(CEL_TOT, REGULY, stan)

def tot_search(start, szerokosc=None, glebokosc=5):
    poziom = [start]
    wygenerowane = 0
    for _ in range(glebokosc):
        nastepny = set()
        for stan in poziom:
            for n in generuj(stan):
                wygenerowane += 1
                if cel_osiagniety(n):
                    return n, wygenerowane
                nastepny.add(n)
        poziom = sorted(nastepny, key=lambda s: (-ocen(s), sorted(s)))
        if szerokosc:
            poziom = poziom[:szerokosc]
    return None, wygenerowane

START_TOT = frozenset({'wysoka_kwota'})
for szer in [None, 3, 1]:
    wynik, n = tot_search(START_TOT, szerokosc=szer)
    print('szerokość:', szer, '| wygenerowane węzły:', n, '| znaleziono:', wynik is not None,
          '| dodane cechy:', sorted(wynik - START_TOT) if wynik else None)


## Wnioski

- Optymalność: BFS zawsze daje najkrótszą ścieżkę. DFS dał tę samą długość tylko tam, gdzie droga jest praktycznie jedna (przykładowy i korytarz). W labiryncie otwartym ścieżka DFS ma 28 kroków wobec 14, a w pułapce 22 wobec 2.
- Koszt: DFS rozwinął mniej węzłów w labiryncie przykładowym i otwartym (23 wobec 29 i 43 wobec 64), ale więcej w pułapce (33 wobec 7) i korytarzu (32 wobec 23). Wynik DFS zależy od kolejności sąsiadów.
- Pamięć: mierzone maksymalnym rozmiarem frontier, DFS zużył tyle samo lub więcej niż BFS (np. 29 wobec 8 w labiryncie otwartym). Wynika to z oznaczania węzłów jako odwiedzone przy dodawaniu do frontier, więc stos trzyma wszystkich sąsiadów. Przewaga DFS z teorii (O(b·m) wobec O(b^d)) w tym pomiarze się nie ujawniła. Do pamięci nie liczę słownika odwiedzonych.
- A* z odległością Manhattan daje taką samą długość ścieżki jak BFS. Rozwinął mniej węzłów w pułapce (3 wobec 7) i minimalnie mniej w przykładowym (28 wobec 29). W labiryncie otwartym i korytarzu tyle samo co BFS, bo w otwartym wszystkie węzły mają tę samą wartość f.
- Heurystyka przeszacowana (waga 10) na czterech labiryntach z G3 nic nie zmieniła, ale w osobnym labiryncie dała ścieżkę 29 kroków zamiast 15. Dopuszczalność heurystyki jest więc warunkiem optymalności.
- Backward chaining zapisany jako DFS na stosie celów daje te same wyniki co wersja rekurencyjna z M3 (470 zbiorów faktów, 25 celów).
- Tree of Thoughts w miniaturze: z ograniczeniem szerokości do 3 wygenerowano 31 węzłów zamiast 66 w pełnym BFS, a do 1 tylko 17, przy tym samym wyniku. Tu funkcja oceny dobrze wskazuje właściwe cechy, w ogólnym przypadku przycinanie może zgubić rozwiązanie.


## Użycie AI

Użyte narzędzie: Claude Code. Zastosowanie: pomoc przy pisaniu kodu, testów i opisów w tym notebooku.

Kontrola: wszystkie komórki uruchomione od początku do końca. A* z dopuszczalną heurystyką daje taką samą długość ścieżki jak BFS, a backward chaining jako DFS daje te same wyniki co wersja rekurencyjna (470 zbiorów faktów, 25 celów). Wyniki i wykresy przejrzane ręcznie.

W P1 generowanie myśli i ocena to zwykłe funkcje, bez wywołania LLM. Reguły z M3 użyte w B2 i P1 są przykładowe i nie były sprawdzane na danych IEEE-CIS.
